# 11.3 - LangGraph Deep Dive

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook rebuilds an AI agent as a **graph** instead of a `while` loop, one LangGraph primitive at a time: a StateGraph that executes, reducers that merge colliding state updates, conditional edges for routing, cycles for the agent loop, checkpointing for memory and resume, and parallel fan-out with the `Send` API. Every example is real, runnable LangGraph with no model call and no API key - you watch the machinery, not the model.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install LangGraph

LangGraph, LangChain, and the Anthropic/core packages are the only dependencies this lesson touches. The install line is commented out - uncomment it the first time you run on Colab or a fresh environment, then leave it commented so re-runs stay fast.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install langchain langchain-anthropic langchain-core langgraph python-dotenv -q  # noqa


**Explanation**

Environment prep, not logic. This cell just documents (and optionally installs) the packages the rest of the notebook imports.

**How the code works - step by step**
- **`langgraph`** - the StateGraph runtime, checkpointers, and the `Send` API this whole lesson exercises.
- **`langchain` / `langchain-core`** - message types (`HumanMessage`, `AIMessage`) used to build and inspect graph state.
- **`langchain-anthropic`** - the `ChatAnthropic` wrapper, only if you extend these graphs with a live model call (the runnable cells here are keyless).
- **`python-dotenv`** - loads keys from a `.env` file so nothing is hardcoded.

**In one line:** uncomment once to install, then keep it quiet.

**What you'll see:** (no output - environment prep)

## Setup - load API keys from the environment

The notebook is deliberately keyless - every runnable cell uses plain functions, not model calls - but this cell wires up the three provider keys so the illustrative Step 7 (and your own experiments) can find them without hardcoding anything.

> **Optional keys** - set any of `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY` to run the illustrative model example; none are needed for Steps 1-6.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Configuration, not computation. `setdefault` seeds each key to an empty string only if it is not already present, so a key already exported in your shell is left untouched.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** - registers the key slot without overwriting an existing value.
- **The other two `setdefault` calls** - same pattern for the Anthropic and Google keys.

**In one line:** reserve the key slots without ever hardcoding a secret.

**What you'll see:** (no output - environment prep)

## 1 - A graph that executes: StateGraph

LangGraph's core object is the **StateGraph**: define a typed **state** every node shares, add **nodes** (functions that read state and return a partial update), wire **edges** from `START` to `END`, then `compile()` into a runnable app you `invoke()`. That is the entire model - a flowchart that runs. Here two plain functions, `parse` then `respond`, run in sequence with no model at all so you can watch the state and the path flow through.

In [ ]:
# A STATEGRAPH IS A FLOWCHART THAT EXECUTES: typed State flows through Nodes along Edges.
# No LLM here - plain functions - so you see the machine, not the model.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):          # the shared state every node reads and writes
    query: str
    path: list
    answer: str

def parse(state):                # a NODE = a function of state -> a PARTIAL state update
    return {"query": state["query"].strip().lower(), "path": state["path"] + ["parse"]}

def respond(state):
    price = {"genai": 14999, "python": 9999}.get(state["query"], 0)
    return {"answer": f"{price} INR", "path": state["path"] + ["respond"]}

graph = StateGraph(State)
graph.add_node("parse", parse)
graph.add_node("respond", respond)
graph.add_edge(START, "parse")      # EDGES wire the flow: START -> parse -> respond -> END
graph.add_edge("parse", "respond")
graph.add_edge("respond", END)
app = graph.compile()               # compile once, invoke many times

out = app.invoke({"query": "  GenAI  ", "path": [], "answer": ""})
print("path:  ", " -> ".join(["START"] + out["path"] + ["END"]))
print("answer:", out["answer"])

# Output:
# path:   START -> parse -> respond -> END
# answer: 14999 INR

**Explanation**

This is the skeleton the rest of the lesson hangs on: state + nodes + edges, compiled once and invoked. The key thing to notice is that each node returns only a *partial* dict - it touches the keys it changes and leaves the rest alone, which is exactly why the graph, not the node, owns the merge.

**How the code works - step by step**
- **`class State(TypedDict)`** - declares the three-key shape (`query`, `path`, `answer`) every node reads and writes.
- **`parse`** - normalizes the query and appends `"parse"` to `path`; returns only those two keys.
- **`respond`** - looks the cleaned query up in a tiny price dict and returns `answer` plus an updated `path`.
- **`add_node` / `add_edge`** - registers the two functions and wires `START -> parse -> respond -> END`.
- **`compile()` then `invoke(...)`** - builds the runnable app and runs one pass over the seed state.

**In one line:** typed state flows START -> parse -> respond -> END, and each node returns a partial update.

**What you'll see:** Prints the executed path `START -> parse -> respond -> END` and `answer: 14999 INR` - the price looked up for the normalized query "genai".

## 2 - State and reducers: the merge problem

This is the concept that trips up every beginner. Nodes return partial updates and the graph merges them - but what if two nodes write the *same* key in one step? The **reducer** annotated on that key decides. Three behaviors matter: **no reducer** is last-write-wins and *crashes* on a concurrent write; **`operator.add`** concatenates; and (next cell) **`add_messages`** dedups by id. This cell proves the first two by running two nodes in parallel.

In [ ]:
# THE MERGE PROBLEM: when two nodes update the SAME key in one step, who wins? A REDUCER decides.
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.errors import InvalidUpdateError

def run_two_parallel(state_type, seed):
    def a(s): return {"out": ["from_a"]}
    def b(s): return {"out": ["from_b"]}
    g = StateGraph(state_type)
    g.add_node("a", a); g.add_node("b", b)
    g.add_edge(START, "a"); g.add_edge(START, "b")   # a and b run IN PARALLEL (same step)
    g.add_edge("a", END); g.add_edge("b", END)
    return g.compile().invoke(seed)

class WithReducer(TypedDict):
    out: Annotated[list, operator.add]        # operator.add: concatenate both updates

print("with operator.add ->", run_two_parallel(WithReducer, {"out": []})["out"])

class NoReducer(TypedDict):
    out: list                                 # no reducer: two writes in one step = crash

try:
    run_two_parallel(NoReducer, {"out": []})
except InvalidUpdateError:
    print("without a reducer ->", "InvalidUpdateError: can receive only one value per step")

# Output:
# with operator.add -> ['from_a', 'from_b']
# without a reducer -> InvalidUpdateError: can receive only one value per step

**Explanation**

A controlled experiment, not an agent: `run_two_parallel` builds the same two-node parallel graph twice, changing only the state type, so the reducer is the single variable. It shows that the reducer is not optional decoration - it is what makes a concurrent write defined at all.

**How the code works - step by step**
- **`run_two_parallel`** - a helper that wires `START -> a` and `START -> b` (so `a` and `b` run in the same step), then invokes the compiled graph.
- **`WithReducer`** - annotates `out` with `Annotated[list, operator.add]`, so the two parallel writes concatenate.
- **First `print`** - runs the reducer version and shows both writes survived.
- **`NoReducer`** - the identical key with no annotation.
- **`try/except InvalidUpdateError`** - runs the no-reducer version and catches the crash the undefined merge raises.

**In one line:** a shared key written in parallel needs a reducer, or the merge is undefined and throws.

**What you'll see:** Prints `with operator.add -> ['from_a', 'from_b']` (both writes concatenated) and `without a reducer -> InvalidUpdateError: can receive only one value per step` (the same graph crashes with no reducer).

## 3 - add_messages: the chat-aware reducer

`operator.add` blindly concatenates, which duplicates a message if you re-send it. **`add_messages`** is the reducer built for the `messages` key: it appends new messages but *dedups by message id*, so a message re-sent with an existing id **replaces** the old one in place. That in-place replace is exactly what lets a human edit a past turn without leaving a duplicate.

In [ ]:
# add_messages: the reducer built for CHAT. It appends new messages, but a message whose id
# already exists REPLACES the old one (dedup) - which is what makes human message-editing work.
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

base = [HumanMessage(content="hi", id="1"), AIMessage(content="hello", id="2")]
appended = add_messages(base, [AIMessage(content="how can I help?", id="3")])  # new id -> append
edited   = add_messages(base, [HumanMessage(content="hi (edited)", id="1")])   # same id -> replace
print("append (new id 3):", [m.content for m in appended])
print("edit   (reuse id 1):", [m.content for m in edited])

# Output:
# append (new id 3): ['hi', 'hello', 'how can I help?']
# edit   (reuse id 1): ['hi (edited)', 'hello']

**Explanation**

A two-case demonstration of one reducer function called directly (no graph needed). Case one adds a message with a fresh id; case two reuses an existing id. The contrast is the whole point: same function, opposite effect, decided entirely by whether the id already exists.

**How the code works - step by step**
- **`base`** - a starting list of two messages with ids `"1"` and `"2"`.
- **`appended`** - calls `add_messages(base, [AIMessage(..., id="3")])`; the new id lands at the end.
- **`edited`** - calls `add_messages(base, [HumanMessage(..., id="1")])`; the existing id `"1"` replaces the original in place.
- **The two `print`s** - list the resulting contents so you can see append vs replace.

**In one line:** new id appends, existing id replaces - use `add_messages` for `messages`, never `operator.add`.

**What you'll see:** Prints `append (new id 3): ['hi', 'hello', 'how can I help?']` and `edit (reuse id 1): ['hi (edited)', 'hello']` - the reused id swaps the old message instead of adding one.

## 4 - Conditional edges and routing

A plain edge always goes to the same next node; a **conditional edge** asks a function which way to go. You attach it with `add_conditional_edges(source, routing_fn, mapping)`: the routing function reads state and returns a key, and the mapping sends that key to a node. This is 11.2's routing pattern as a graph - a cheap keyword classifier picks `billing` / `technical` / `general` and lights exactly one branch, no LLM required.

In [ ]:
# CONDITIONAL EDGES: a routing function reads state and NAMES the next node. Branch as a graph.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    query: str
    route: str
    answer: str

def classify(state):                              # a cheap keyword router (no LLM)
    q = state["query"].lower()
    if any(w in q for w in ("refund", "price", "emi")): return {"route": "billing"}
    if any(w in q for w in ("python", "module", "prereq")): return {"route": "technical"}
    return {"route": "general"}

def billing(state):   return {"answer": "[billing] 7-day full refund; GenAI 14999 INR"}
def technical(state): return {"answer": "[technical] prereqs: Python + HS math; 14 modules"}
def general(state):   return {"answer": "[general] how can I help?"}

def route_decision(state): return state["route"]  # returns the NAME of the next node

g = StateGraph(State)
for name, fn in [("classify", classify), ("billing", billing),
                 ("technical", technical), ("general", general)]:
    g.add_node(name, fn)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route_decision,        # map route -> node
                        {"billing": "billing", "technical": "technical", "general": "general"})
for n in ("billing", "technical", "general"): g.add_edge(n, END)
app = g.compile()

for q in ["Can I get a refund?", "What Python version?", "Hello!"]:
    out = app.invoke({"query": q, "route": "", "answer": ""})
    print(f"{q:24s} -> {out['route']:9s} -> {out['answer']}")

# Output:
# Can I get a refund?      -> billing   -> [billing] 7-day full refund; GenAI 14999 INR
# What Python version?     -> technical -> [technical] prereqs: Python + HS math; 14 modules
# Hello!                   -> general   -> [general] how can I help?

**Explanation**

Shows branching as a first-class graph feature. The design separates two jobs: `classify` writes a decision into state, and `route_decision` reads that decision back out to name the next node - keeping the routing logic and the state update cleanly split.

**How the code works - step by step**
- **`classify`** - a keyword router that inspects the query and writes `{"route": ...}` into state (no model).
- **`billing` / `technical` / `general`** - the three destination nodes, each returning a canned answer.
- **`route_decision`** - the routing function; it simply returns `state["route"]`, the name of the next node.
- **`add_conditional_edges("classify", route_decision, {...})`** - maps each route name to its node.
- **The `for` loop** - invokes three sample queries and prints which branch each one lit.

**In one line:** a routing function reads state and names the next node, so the graph branches.

**What you'll see:** Prints one line per query showing the route and answer - e.g. `Can I get a refund? -> billing -> [billing] 7-day full refund...`, `What Python version? -> technical -> ...`, `Hello! -> general -> ...`.

## 5 - Cycles: the agent loop as a graph

Every graph so far ran forward to `END`. An agent needs a **cycle**: call the model, and if it asked for a tool, run the tool and go *back* to the model. That is a conditional edge (`should_continue`) routing to `tools` or `END`, plus a plain edge from `tools` back to `agent`. It is the observe-think-act loop from 11.1 drawn as a graph - the agent node is scripted so it runs keyless - and a **`recursion_limit`** guarantees a stuck cycle cannot spin forever.

In [ ]:
# THE AGENT LOOP AS A GRAPH: agent -> (tool_calls?) -> tools -> back to agent, until done.
# The agent node is SCRIPTED so this runs with no key, but the graph is real LangGraph.
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    task: str
    scratch: Annotated[list, operator.add]   # accumulates think/act/observe lines
    answer: str

KB = {"genai": "14999"}
def tool(name, arg):
    if name == "search":    return KB.get(arg, "not found")
    if name == "calculate": return str(round(eval(arg, {"__builtins__": {}}), 2))
    return "unknown tool"

def agent(state):                            # SCRIPTED think step (a real LLM would decide this)
    acts = [s for s in state["scratch"] if s.startswith("act")]
    if not any("search" in a for a in acts):
        return {"scratch": ["think: need the price", "act: search(genai)"]}
    if not any("calculate" in a for a in acts):
        return {"scratch": ["think: add 18% GST", "act: calculate(14999*1.18)"]}
    return {"answer": "17698.82 INR", "scratch": ["think: done"]}

def tools(state):
    last = state["scratch"][-1]              # e.g. "act: search(genai)"
    call = last.split("act:")[1].strip()
    name, arg = call[:-1].split("(")
    return {"scratch": [f"observe: {tool(name, arg)}"]}

def should_continue(state):                  # conditional edge: loop or stop
    return "tools" if state["scratch"][-1].startswith("act") else "end"

g = StateGraph(State)
g.add_node("agent", agent); g.add_node("tools", tools)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
g.add_edge("tools", "agent")                 # the CYCLE: tools -> agent
app = g.compile()

out = app.invoke({"task": "GenAI price with 18% GST?", "scratch": [], "answer": ""},
                 {"recursion_limit": 12})    # a hard cap so a stuck graph cannot spin forever
for line in out["scratch"]: print(" ", line)
print("answer:", out["answer"])

# Output:
#   think: need the price
#   act: search(genai)
#   observe: 14999
#   think: add 18% GST
#   act: calculate(14999*1.18)
#   observe: 17698.82
#   think: done
# answer: 17698.82 INR

**Explanation**

This is the first graph with a loop in it, and it makes the agent pattern concrete: the cycle is just an edge that points backwards. The agent's decisions are hard-scripted (a real LLM would make them), so the cell isolates the *control flow* - routing, looping, and the safety cap - from the model.

**How the code works - step by step**
- **`State`** - `scratch` is annotated with `operator.add` so each node's think/act/observe lines accumulate.
- **`tool` / `KB`** - a tiny search+calculate backend the tools node calls.
- **`agent`** - the scripted think step: decides to search, then to calculate, then to finish, based on what is already in `scratch`.
- **`tools`** - parses the last `act:` line, runs the named tool, and appends an `observe:` line.
- **`should_continue`** - the conditional edge: return `"tools"` if the last line is an action, else `"end"`.
- **`add_edge("tools", "agent")`** - the back-edge that closes the cycle.
- **`invoke(..., {"recursion_limit": 12})`** - runs the loop with a hard cap so it cannot run forever.

**In one line:** agent routes to tools, tools route back, and `recursion_limit` is the backstop.

**What you'll see:** Prints the full think/act/observe trace (search genai -> 14999, calculate 14999*1.18 -> 17698.82, done) followed by `answer: 17698.82 INR`.

## 6 - Persistence and checkpointing

This is LangGraph's killer feature. Compile the graph with a **checkpointer** and pass a **`thread_id`**, and the runtime autosaves the full state after every step. Three things fall out for free: **memory** (invoke the same thread again and prior messages are still there), **resume** (restart after a crash from the last checkpoint), and **time-travel** (`get_state_history` returns every checkpoint to inspect or fork from). Use `InMemorySaver` for dev, `SqliteSaver`/`PostgresSaver` in production.

In [ ]:
# PERSISTENCE: compile with a checkpointer + pass a thread_id, and state SURVIVES between calls.
# This is how a graph gets memory, resume-after-crash, and time-travel - for free.
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver   # swap for SqliteSaver / PostgresSaver in prod
from langchain_core.messages import HumanMessage, AIMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]

def turn(state):                             # canned reply (no LLM) so the run is deterministic
    user = state["messages"][-1].content
    reply = "GenAI course is 14999 INR." if "price" in user.lower() else "With 18% GST: 17698.82 INR."
    return {"messages": [AIMessage(content=reply)]}

g = StateGraph(State); g.add_node("turn", turn)
g.add_edge(START, "turn"); g.add_edge("turn", END)
app = g.compile(checkpointer=InMemorySaver())            # <-- the one line that adds memory

cfg = {"configurable": {"thread_id": "user-123"}}
app.invoke({"messages": [HumanMessage(content="GenAI price?")]}, cfg)   # turn 1
app.invoke({"messages": [HumanMessage(content="With 18% GST?")]}, cfg)  # turn 2, SAME thread

state = app.get_state(cfg)
print("thread user-123 remembers", len(state.values["messages"]), "messages:")
for m in state.values["messages"]: print("   ", type(m).__name__, "-", m.content)
print("checkpoints saved (time-travel):", len(list(app.get_state_history(cfg))))

fresh = app.get_state({"configurable": {"thread_id": "user-999"}})
print("a NEW thread starts empty:", fresh.values.get("messages", []))

# Output:
# thread user-123 remembers 4 messages:
#     HumanMessage - GenAI price?
#     AIMessage - GenAI course is 14999 INR.
#     HumanMessage - With 18% GST?
#     AIMessage - With 18% GST: 17698.82 INR.
# checkpoints saved (time-travel): 6
# a NEW thread starts empty: []

**Explanation**

Demonstrates that persistence is a one-line compile-time change, not a rewrite. The graph itself is a trivial single-node echo; the point is entirely in `compile(checkpointer=...)` plus the `thread_id` config, and in showing that two different threads are fully isolated.

**How the code works - step by step**
- **`State` with `add_messages`** - the message list uses the chat reducer from Step 3.
- **`turn`** - a canned reply node (no LLM) so the run is deterministic.
- **`compile(checkpointer=InMemorySaver())`** - the single line that turns on autosave.
- **Two `invoke`s with the same `cfg` (`thread_id: user-123`)** - turn 2 sees turn 1's history.
- **`get_state(cfg)`** - reads back the accumulated messages for that thread.
- **`get_state_history(cfg)`** - counts the saved checkpoints available for time-travel.
- **`get_state({thread_id: user-999})`** - a fresh thread, shown to start empty (threads are isolated).

**In one line:** a checkpointer + `thread_id` buys memory, resume, and time-travel for one line of setup.

**What you'll see:** Prints that thread `user-123` remembers 4 messages (listing both human turns and both AI replies), `checkpoints saved (time-travel): 6`, and that new thread `user-999` starts empty (`[]`).

## 7 - Parallelism: fan-out and the Send API

Because the graph knows the whole structure, it can run independent nodes at the same time. **Static fan-out:** two edges out of one node both run in the same step, and a downstream node with two incoming edges waits for both (fan-in), a reducer merging their output. **Dynamic fan-out (the `Send` API):** when the number of parallel tasks is unknown until runtime, a routing function returns a list of `Send("node", state)` objects and LangGraph launches one worker each. That is clean map-reduce - map with `Send`, reduce with a reducer.

In [ ]:
# PARALLELISM: multiple edges out of a node run IN PARALLEL; Send() fans out DYNAMICALLY.
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# -- Static fan-out / fan-in: two nodes run together, a reducer merges their results --
class FanState(TypedDict):
    facts: Annotated[list, operator.add]
def price_lookup(s):  return {"facts": ["price: 14999"]}
def hours_lookup(s):  return {"facts": ["hours: 146"]}
def summarize(s):     return {"facts": ["summary: " + " | ".join(sorted(s["facts"]))]}
g = StateGraph(FanState)
g.add_node("price", price_lookup); g.add_node("hours", hours_lookup); g.add_node("sum", summarize)
g.add_edge(START, "price"); g.add_edge(START, "hours")   # fan-out
g.add_edge("price", "sum");  g.add_edge("hours", "sum")   # fan-in (sum waits for BOTH)
g.add_edge("sum", END)
print("fan-out/fan-in:", g.compile().invoke({"facts": []})["facts"][-1])

# -- Dynamic map-reduce with Send: number of workers unknown until runtime --
class MapState(TypedDict):
    docs: list
    scored: Annotated[list, operator.add]
def fan_out(state):                          # returns one Send per doc -> N parallel workers
    return [Send("score", {"doc": d}) for d in state["docs"]]
class WorkerState(TypedDict):
    doc: str
    scored: Annotated[list, operator.add]
def score(state): return {"scored": [f"{state['doc']}:{len(state['doc'])}"]}
m = StateGraph(MapState)
m.add_node("score", score)
# NOTE: unlike Step 3's DICT mapping, the 3rd arg here is just the LIST of possible
# target nodes - fan_out returns Send objects directly, not a route key to look up.
m.add_conditional_edges(START, fan_out, ["score"])
m.add_edge("score", END)
print("Send map-reduce:", sorted(m.compile().invoke({"docs": ["genai", "python", "sql"], "scored": []})["scored"]))

# Output:
# fan-out/fan-in: summary: hours: 146 | price: 14999
# Send map-reduce: ['genai:5', 'python:6', 'sql:3']

**Explanation**

Two mini-graphs in one cell that contrast the two flavors of parallelism. The static half shows fan-out/fan-in with fixed structure; the dynamic half shows `Send` deciding the worker count from the input list. Both lean on the same rule from Step 2 - every parallel-written key must carry a reducer.

**How the code works - step by step**
- **`FanState` + `price_lookup` / `hours_lookup` / `summarize`** - the static graph; `facts` uses `operator.add`.
- **`add_edge(START, "price")` and `add_edge(START, "hours")`** - static fan-out (both run together).
- **`add_edge("price", "sum")` and `add_edge("hours", "sum")`** - fan-in; `sum` waits for both branches.
- **`MapState` + `fan_out`** - the dynamic graph; `fan_out` returns one `Send("score", {"doc": d})` per document.
- **`add_conditional_edges(START, fan_out, ["score"])`** - note the third arg is a *list* of target nodes, not Step 4's dict, because `fan_out` returns `Send` objects directly rather than a route key.
- **`score`** - a worker that writes to the reducer-annotated `scored` key so results merge.

**In one line:** structure gives you parallelism (static edges or `Send`), and a reducer safely fans the results back in.

**What you'll see:** Prints `fan-out/fan-in: summary: hours: 146 | price: 14999` (both static branches merged) and `Send map-reduce: ['genai:5', 'python:6', 'sql:3']` (one worker per document, results reduced).

Across seven primitives you turned a hand-rolled agent loop into a graph: typed state through nodes and edges (Step 1), reducers that decide how colliding updates merge (Step 2), the chat-aware `add_messages` reducer (Step 3), conditional edges that branch by state (Step 4), a cycle capped by `recursion_limit` for the agent loop (Step 5), a checkpointer plus `thread_id` for memory and time-travel (Step 6), and static/dynamic fan-out with `Send` for parallel map-reduce (Step 7). The graph model is what separates a demo loop from a production agent: checkpointing, branching, parallelism, and human pauses all fall out of the structure. Next up: memory systems beyond thread state (11.4), multi-agent coordination (11.5), and deep human-in-the-loop with `interrupt()` (11.10).